# Classification Report Demo

## Setup Data

In [8]:
from pathlib import Path
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from mlreport import ComparisonReport, Report

Path("reports").mkdir(exist_ok=True)

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
author = "Lucas Summers"
theme = "light"

## Model 1: Train/Test Split Report

In [9]:
split_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

split_model.fit(X_train, y_train)
y_pred_train = split_model.predict(X_train)
y_pred_test = split_model.predict(X_test)

split_report = Report(
    split_model,
    title="Iris Classification Report - Train/Test Split",
    author=author,
    description="Logistic Regression evaluated on a standard train/test split.",
    theme=theme,
)

split_report.add_split("train", X_train, y_train, y_pred_train)
split_report.add_split("test", X_test, y_test, y_pred_test)
split_report.build().to_pdf("reports/iris_split_report.pdf").summary()

Report
  title: Iris Classification Report - Train/Test Split
  author: Lucas Summers
  description: Logistic Regression evaluated on a standard train/test split.
  generated_at: 2026-04-17 01:09

Model
  name: LogisticRegression
  type: Classification
  sklearn_version: 1.6.1
  param_count: 23

Data
  features: 4
  splits: train=112, test=38
  total: 150

Metrics
  Accuracy: train=0.9643, test=0.9211
  F1 Score (Macro): train=0.9640, test=0.9230
  F1 Score (Weighted): train=0.9643, test=0.9209
  Precision (Macro): train=0.9640, test=0.9246
  Precision (Weighted): train=0.9643, test=0.9226
  Recall (Macro): train=0.9640, test=0.9231
  Recall (Weighted): train=0.9643, test=0.9211


## Model 2: CV Report

In [10]:
cv_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=4,
    random_state=42,
)
cv_model.fit(X_train, y_train)


cv_report = Report(
    cv_model,
    title="Iris Classification Report - Cross-Validation",
    author=author,
    description="Random Forest evaluated with 5-fold stratified cross-validation.",
    theme=theme,
)

cv_report.add_crossval(X_train, y_train, cv=cv)
cv_report.build().to_pdf("reports/iris_cv_report.pdf").summary()

Report
  title: Iris Classification Report - Cross-Validation
  author: Lucas Summers
  description: Random Forest evaluated with 5-fold stratified cross-validation.
  generated_at: 2026-04-17 01:09

Model
  name: RandomForestClassifier
  type: Classification
  sklearn_version: 1.6.1
  param_count: 19

Data
  features: 4
  splits: cv=112
  total: 112
  cv_folds: 5

Metrics
  Accuracy: cv=0.9644 (std=0.0328)
  F1 Score (Macro): cv=0.9629 (std=0.0347)
  F1 Score (Weighted): cv=0.9638 (std=0.0336)
  Precision (Macro): cv=0.9709 (std=0.0258)
  Precision (Weighted): cv=0.9701 (std=0.0267)
  Recall (Macro): cv=0.9619 (std=0.0356)
  Recall (Weighted): cv=0.9644 (std=0.0328)


## Model 3: Tuning + CV Report

In [11]:
search = GridSearchCV(
    estimator=Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier()),
        ]
    ),
    param_grid={
        "model__n_neighbors": [3, 5, 7, 9],
        "model__metric": ["euclidean", "manhattan"],
    },
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

search.fit(X_train, y_train)
best_tuned_model = search.best_estimator_

tuned_cv_report = Report(
    best_tuned_model,
    title="Iris Classification Report - Tuning + Cross-Validation",
    author=author,
    description="KNN tuned with GridSearchCV and evaluated with 5-fold CV.",
    theme=theme,
)

tuned_cv_report.add_crossval(X_train, y_train, cv=cv)
tuned_cv_report.add_search(search)
tuned_cv_report.build().to_pdf("reports/iris_tuned_cv_report.pdf").summary()

Report
  title: Iris Classification Report - Tuning + Cross-Validation
  author: Lucas Summers
  description: KNN tuned with GridSearchCV and evaluated with 5-fold CV.
  generated_at: 2026-04-17 01:09

Model
  name: KNeighborsClassifier
  type: Classification
  sklearn_version: 1.6.1
  param_count: 16
  tuning_method: GridSearchCV
  tuning_metric: accuracy
  tuning_best_score: 0.9731
  tuning_candidates: 8
  tuning_cv_folds: 5
  tuning_best_params: model__metric=euclidean, model__n_neighbors=3

Data
  features: 4
  splits: cv=112
  total: 112
  cv_folds: 5

Metrics
  Accuracy: cv=0.9731 (std=0.0220)
  F1 Score (Macro): cv=0.9730 (std=0.0220)
  F1 Score (Weighted): cv=0.9730 (std=0.0221)
  Precision (Macro): cv=0.9769 (std=0.0190)
  Precision (Weighted): cv=0.9762 (std=0.0194)
  Recall (Macro): cv=0.9726 (std=0.0225)
  Recall (Weighted): cv=0.9731 (std=0.0220)


## Comparison Report of 3 Models

In [12]:
comparison = ComparisonReport(
    reports=[
        split_report,
        cv_report,
        tuned_cv_report,
    ],
    title="Iris Workflow Comparison",
    author=author,
    description="Comparison of three classification workflows on the Iris dataset.",
    theme=theme,
)

comparison.build().to_pdf("reports/iris_comparison_report.pdf").summary()

Comparison Report
  title: Iris Workflow Comparison
  author: Lucas Summers
  description: Comparison of three classification workflows on the Iris dataset.
  baseline: LogisticRegression

Models
  1. LogisticRegression (LogisticRegression) [baseline]: Logistic Regression evaluated on a standard train/test split.
  2. RandomForestClassifier (RandomForestClassifier): Random Forest evaluated with 5-fold stratified cross-validation.
  3. KNeighborsClassifier (KNeighborsClassifier): KNN tuned with GridSearchCV and evaluated with 5-fold CV.

Metrics
  Accuracy: LogisticRegression=0.9211, RandomForestClassifier=0.9644 (+0.0434), KNeighborsClassifier=0.9731 (+0.0521), best=KNeighborsClassifier
  F1 Score (Macro): LogisticRegression=0.9230, RandomForestClassifier=0.9629 (+0.0399), KNeighborsClassifier=0.9730 (+0.0500), best=KNeighborsClassifier
  F1 Score (Weighted): LogisticRegression=0.9209, RandomForestClassifier=0.9638 (+0.0429), KNeighborsClassifier=0.9730 (+0.0520), best=KNeighborsClassi